# SC4002 Natural Language Processing AY26-27 Sem1 Group Assignment

## Group 10

| Name | StudentID |
|---|---|
| Koh Wee Xuan | U |
| Seow Jia Wei Jerrald | U2322282A |
| Woo Weng Tai | U |
| Tan Jin Yong | U2322914J |
| Tan Zhe Kai | U2322419A |

## Part A: Data loading, preprocessing and shared evaluation

This part covers the following steps:

1. **Load and verify the dataset**: load MASSIVE 1.0 `en-US.jsonl`, split by the original `partition` field, and assert the split sizes.
2. **Implement preprocessing**: convert each raw record into model inputs (token ids) and labels (intent id + one BIO slot tag per token).
3. **Implement the shared evaluation functions**: intent accuracy and slot micro-F1 with one fixed set of scoring rules, plus a quick check that they are correct.

### Setup

The notebook expects the official archive `amazon-massive-dataset-1.0.tar.gz` in the same folder as this notebook. On the first run, `en-US.jsonl` is extracted from it.

In [2]:
import json
import re
import tarfile
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

In [3]:
# dataset locations
DATA_ROOT    = Path(".")                            # same folder as this notebook and the .tar.gz
ARCHIVE_NAME = "amazon-massive-dataset-1.0.tar.gz"
MEMBER       = "1.0/data/en-US.jsonl"               # path inside the archive

# split sizes
EXPECTED_SPLIT_SIZES = {"train": 11_514, "dev": 2_033, "test": 2_974}
EXPECTED_TOTAL       = 16_521

# preprocessing settings (the vocab is built from train only)
MIN_FREQ = 1    # words seen fewer times than this in train become <unk>
SEED     = 42   # keeps the training shuffle order the same every run

## 1. Load and verify the dataset

### 1.1 Extract `en-US.jsonl`

In [5]:
# only pull the one file we need out of the archive, and only if it isn't there yet
DATA_FILE = DATA_ROOT / MEMBER
if not DATA_FILE.is_file():
    with tarfile.open(DATA_ROOT / ARCHIVE_NAME, "r:gz") as tar:
        safe = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}  
        tar.extract(MEMBER, path=DATA_ROOT, **safe)
print(f"data file: {DATA_FILE.as_posix()}")

data file: 1.0/data/en-US.jsonl


### 1.2 Load the records and partition them by the original `partition` field

The examples are kept exactly as released. None of the data is re-split, added, filtered or de-duplicated.

In [6]:
# load every record, then peek at the first few rows
with DATA_FILE.open(encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)
df.head()

,id,locale,partition,scenario,intent,utt,annot_utt,worker_id
0,0,en-US,test,alarm,alarm_set,wake me up at five am this week,wake me up at [time : five am] [date : this week],1
1,1,en-US,train,alarm,alarm_set,wake me up at nine am on friday,wake me up at [time : nine am] on [date : friday],1
2,2,en-US,train,alarm,alarm_set,set an alarm for two hours from now,set an alarm for [time : two hours from now],1
3,3,en-US,test,audio,audio_volume_mute,quiet,quiet,1
4,4,en-US,train,audio,audio_volume_mute,olly quiet,olly quiet,1


In [7]:
raw_splits = {name: [r for r in records if r["partition"] == name] for name in EXPECTED_SPLIT_SIZES}

print(f"Total records: {len(records):,}")
for name, expected in EXPECTED_SPLIT_SIZES.items():
    print(f"  {name:<5}: {len(raw_splits[name]):>6,} examples (expected {expected:,})")

# stop here if the split sizes don't match the brief
assert len(records) == EXPECTED_TOTAL, f"expected {EXPECTED_TOTAL} records, got {len(records)}"
for name, expected in EXPECTED_SPLIT_SIZES.items():
    assert len(raw_splits[name]) == expected, f"{name}: expected {expected}, got {len(raw_splits[name])}"
print("\n[OK] Split sizes match: 11,514 train / 2,033 dev / 2,974 test (16,521 total)")

Total records: 16,521
  train: 11,514 examples (expected 11,514)
  dev  :  2,033 examples (expected 2,033)
  test :  2,974 examples (expected 2,974)

[OK] Split sizes match: 11,514 train / 2,033 dev / 2,974 test (16,521 total)


## 2. Data preprocessing

Each raw record becomes:

| field | meaning |
|---|---|
| `tokens`, `offsets` | whitespace tokens of `utt` and their character offsets |
| `bio_tags` | one BIO slot tag per token, e.g. `B-time I-time` |
| `gold_slots` | gold `(slot_type, span_text)` pairs, read straight from `annot_utt` |
| `token_ids`, `tag_ids`, `intent_id` | integer encodings fed to the model |

### 2.1 Parse the annotation, tokenise, and tag

`annot_utt` marks slots as `[slot_type : value]`. Every bracket in MASSIVE en-US has a space on both sides, so splitting `utt` on whitespace always lines slots up with whole tokens. The model only sees `utt`, and the `scenario` field is never used.

In [8]:
SLOT_RE = re.compile(r"\[\s*([a-z_]+)\s*:\s*(.*?)\s*\]")
TOKEN_RE = re.compile(r"\S+")


def parse_annotation(annot_utt: str):
    """strip the slot brackets out of annot_utt.

    e.g. 'wake me up at [time : five am]'  ->  ('wake me up at five am', [('time', 14, 21)])

    gives back the plain text plus each slot as (slot_type, char_start, char_end) inside that text.
    """
    text_parts, slots, cursor, length = [], [], 0, 0
    for m in SLOT_RE.finditer(annot_utt):
        before = annot_utt[cursor:m.start()]
        slot_type, value = m.group(1), m.group(2)
        text_parts += [before, value]
        start = length + len(before)
        slots.append((slot_type, start, start + len(value)))
        length = start + len(value)
        cursor = m.end()
    return "".join(text_parts) + annot_utt[cursor:], slots


def tokenize(utt: str):
    """split utt on whitespace and keep each token's (start, end) character offsets."""
    matches = list(TOKEN_RE.finditer(utt))
    return [m.group() for m in matches], [(m.start(), m.end()) for m in matches]


def spans_to_bio(offsets, char_slots):
    """turn character-level slots into one bio tag per token. errors out if a slot cuts through a token."""
    tags = ["O"] * len(offsets)
    start_tok = {s: i for i, (s, _) in enumerate(offsets)}
    end_tok = {e: i for i, (_, e) in enumerate(offsets)}
    for slot_type, s, e in char_slots:
        if s not in start_tok or e not in end_tok:
            raise ValueError(f"slot boundary inside a token: {(slot_type, s, e)}")
        i, j = start_tok[s], end_tok[e]
        tags[i] = f"B-{slot_type}"
        for k in range(i + 1, j + 1):
            tags[k] = f"I-{slot_type}"
    return tags


def preprocess_record(r):
    """turn one raw record into tokens, offsets, bio tags and the gold (slot_type, span_text) pairs."""
    text, char_slots = parse_annotation(r["annot_utt"])
    if text != r["utt"]:
        raise ValueError(f"annotation text differs from utt: {text!r} vs {r['utt']!r}")
    tokens, offsets = tokenize(r["utt"])
    return {
        "id": r["id"],
        "utt": r["utt"],
        "intent": r["intent"],
        "tokens": tokens,
        "offsets": offsets,
        "bio_tags": spans_to_bio(offsets, char_slots),
        "gold_slots": [(t, r["utt"][s:e]) for t, s, e in char_slots],
    }


data = {name: [preprocess_record(r) for r in rs] for name, rs in raw_splits.items()}
for name in data:
    for i, ex in enumerate(data[name]):
        ex["index"] = i  # position in its split, so batched predictions can be matched back up

### 2.2 Label maps and word vocabulary (built from train only)

Labels are sorted alphabetically so the ids are the same on every machine. The tag set is `O` plus `B-` and `I-` for each of the 55 slot types (111 tags). Padded slot positions get `PAD_TAG_ID = -100`, which `torch.nn.CrossEntropyLoss` skips by default. In the word vocabulary, id 0 is `<pad>` and id 1 is `<unk>`. Dev and test words that never appear in train map to `<unk>`.

In [9]:
intent_itos = sorted({ex["intent"] for ex in data["train"]})
intent_stoi = {lab: i for i, lab in enumerate(intent_itos)}

slot_types = sorted({t for ex in data["train"] for t, _ in ex["gold_slots"]})
tag_itos = ["O"] + [f"{prefix}-{t}" for t in slot_types for prefix in ("B", "I")]
tag_stoi = {tag: i for i, tag in enumerate(tag_itos)}
PAD_TAG_ID = -100


class Vocab:
    """word to id lookup. id 0 is <pad>, id 1 is <unk>, then words from most to least frequent in train."""

    def __init__(self, counter: Counter, min_freq: int = 1):
        words = sorted((w for w, c in counter.items() if c >= min_freq), key=lambda w: (-counter[w], w))
        self.itos = ["<pad>", "<unk>"] + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id, self.unk_id = 0, 1

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(t.lower(), self.unk_id) for t in tokens]


vocab = Vocab(Counter(t.lower() for ex in data["train"] for t in ex["tokens"]), min_freq=MIN_FREQ)

# turn every split into ids
for name in data:
    for ex in data[name]:
        ex["token_ids"] = vocab.encode(ex["tokens"])
        ex["tag_ids"] = [tag_stoi[t] for t in ex["bio_tags"]]
        ex["intent_id"] = intent_stoi[ex["intent"]]

print(f"{len(intent_itos)} intents | {len(tag_itos)} bio tags | vocabulary of {len(vocab):,} words")

60 intents | 111 bio tags | vocabulary of 5,304 words


### 2.3 Batching

Batches are padded to the longest utterance in the batch. Test utterances go up to 61 tokens while train tops out at 35, so a fixed length taken from train would cut test inputs short. Each batch is a dict:

| key | shape | content |
|---|---|---|
| `input_ids` | `(B, T)` | word ids, padded with `vocab.pad_id` (0) |
| `attention_mask` | `(B, T)` | `True` for real tokens |
| `lengths` | `(B,)` | real token count (works with `pack_padded_sequence(..., enforce_sorted=False)`) |
| `intent_labels` | `(B,)` | intent ids |
| `slot_labels` | `(B, T)` | bio tag ids, padded with `PAD_TAG_ID` (-100) |
| `indices` | `(B,)` | position of each example in its split |

In [10]:
def collate_batch(batch):
    """pad every example to the longest one in the batch (the keys are listed in the table above)."""
    lengths = torch.tensor([len(ex["token_ids"]) for ex in batch], dtype=torch.long)
    max_len = int(lengths.max())
    input_ids = torch.full((len(batch), max_len), vocab.pad_id, dtype=torch.long)
    slot_labels = torch.full((len(batch), max_len), PAD_TAG_ID, dtype=torch.long)
    for row, ex in enumerate(batch):
        n = len(ex["token_ids"])
        input_ids[row, :n] = torch.tensor(ex["token_ids"], dtype=torch.long)
        slot_labels[row, :n] = torch.tensor(ex["tag_ids"], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": torch.arange(max_len)[None, :] < lengths[:, None],
        "lengths": lengths,
        "intent_labels": torch.tensor([ex["intent_id"] for ex in batch], dtype=torch.long),
        "slot_labels": slot_labels,
        "indices": torch.tensor([ex["index"] for ex in batch], dtype=torch.long),
    }


def make_loader(split: str, batch_size: int = 32, shuffle: bool = False, seed: int = SEED) -> DataLoader:
    """dataloader for one split. only shuffle train; the seeded generator gives the same order every run."""
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(data[split], batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_batch, generator=generator)

## 3. Implement the shared evaluation functions

Every experiment, Stage 1 and Stage 2, is scored with `evaluate()` and nothing else. A model has to give one prediction per utterance in the form `{"intent": "alarm_set", "slots": [("time", "five am"), ("date", "this week")]}`. For a BIO tagger, `make_prediction_from_ids()` does that conversion.

- **Intent accuracy** = 100 × correct intents / number of utterances.
- **Slot micro-F1** = 100 × 2TP / (2TP + FP + FN), pooled over all utterances. A true positive needs both the slot type and the span text to match exactly.

Scoring rules, fixed for every experiment:

1. Gold slots come straight from `annot_utt`.
2. Slot types and spans are lowercased and extra spaces are collapsed before comparing. A span with a word missing or added is still a miss.
3. Slots are matched per utterance as a multiset. If `[person : james]` appears twice in the gold, predicting it once earns one hit and one miss.
4. An `I-x` tag that doesn't continue a slot of type `x` starts a new slot (the conlleval convention).

`evaluate()` also returns slot precision and recall, plus `combined` = (intent accuracy + slot F1) / 2 in case the group picks checkpoints on both.

In [11]:
def bio_to_spans(tags):
    """bio tags to [(slot_type, start_token, end_token_exclusive)]. broken sequences are handled the conlleval way."""
    spans, cur_type, start = [], None, None
    for i, tag in enumerate(list(tags) + ["O"]):  # the extra outside tag at the end closes any slot still open
        prefix, typ = ("O", None) if tag == "O" else (tag[0], tag[2:])
        if prefix == "I" and typ == cur_type:
            continue                              # same slot, keep going
        if cur_type is not None:
            spans.append((cur_type, start, i))    # close off the current slot
        cur_type, start = (typ, i) if prefix != "O" else (None, None)
    return spans


def make_prediction_from_ids(example, intent_id, tag_ids):
    """turn model output ids (ints or tensors) into the shared prediction format. padding past the utterance is dropped."""
    tags = [tag_itos[int(t)] for t in list(tag_ids)[:len(example["tokens"])]]
    utt, offsets = example["utt"], example["offsets"]
    slots = [(typ, utt[offsets[s][0]:offsets[e - 1][1]]) for typ, s, e in bio_to_spans(tags)]
    return {"intent": intent_itos[int(intent_id)], "slots": slots}


def _normalize_slot(slot):
    slot_type, span = slot
    return slot_type.strip().lower(), " ".join(span.lower().split())


def intent_accuracy(gold_intents, pred_intents) -> float:
    """100 x correct intents / number of utterances."""
    return 100.0 * sum(g == p for g, p in zip(gold_intents, pred_intents)) / len(gold_intents)


def slot_micro_f1(gold_slots, pred_slots) -> dict:
    """exact-match slot micro-f1 = 100 x 2tp / (2tp + fp + fn), pooled over all utterances."""
    tp = fp = fn = 0
    for gold, pred in zip(gold_slots, pred_slots):
        g = Counter(map(_normalize_slot, gold))
        p = Counter(map(_normalize_slot, pred))
        matched = sum((g & p).values())  # multiset overlap, so a slot mentioned twice must be predicted twice
        tp += matched
        fp += sum(p.values()) - matched
        fn += sum(g.values()) - matched
    return {
        "precision": 100.0 * tp / (tp + fp) if tp + fp else 0.0,
        "recall": 100.0 * tp / (tp + fn) if tp + fn else 0.0,
        "f1": 100.0 * 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0,
        "tp": tp, "fp": fp, "fn": fn,
    }


def evaluate(examples, predictions) -> dict:
    """the one scorer every experiment uses. predictions must be in the same order as examples."""
    if len(predictions) != len(examples):
        raise ValueError("need exactly one prediction per example")
    acc = intent_accuracy([ex["intent"] for ex in examples], [p["intent"] for p in predictions])
    slot = slot_micro_f1([ex["gold_slots"] for ex in examples], [p["slots"] for p in predictions])
    return {
        "n": len(examples),
        "intent_accuracy": acc,
        "slot_micro_f1": slot["f1"],
        "slot_precision": slot["precision"],
        "slot_recall": slot["recall"],
        "combined": (acc + slot["f1"]) / 2,
    }


def format_scores(scores: dict, split: str = "") -> str:
    head = f"[{split}] " if split else ""
    return (f"{head}Intent accuracy: {scores['intent_accuracy']:.2f}% | "
            f"Slot micro-F1: {scores['slot_micro_f1']:.2f}% "
            f"(P {scores['slot_precision']:.2f}%, R {scores['slot_recall']:.2f}%) | n={scores['n']:,}")

### 3.1 Quick check

Two hand-worked cases test rules 2 and 3. Then the gold tags are fed through the same path model outputs will take. If parsing, tagging, encoding and scoring all agree, every split scores 100.00 / 100.00.

In [12]:
# one wrong span: tp=1, fp=1, fn=1, so f1 = 50
r = slot_micro_f1([[("time", "five am"), ("date", "this week")]], [[("time", "five am"), ("date", "week")]])
assert (r["tp"], r["fp"], r["fn"]) == (1, 1, 1) and r["f1"] == 50.0

# 'james' twice in gold, once in pred: one hit, one miss
r = slot_micro_f1([[("person", "james"), ("person", "james")]], [[("person", "james")]])
assert (r["tp"], r["fp"], r["fn"]) == (1, 0, 1)

for name in ("train", "dev", "test"):
    gold_as_preds = [make_prediction_from_ids(ex, ex["intent_id"], ex["tag_ids"]) for ex in data[name]]
    scores = evaluate(data[name], gold_as_preds)
    print(format_scores(scores, f"{name} | gold tags"))
    assert scores["intent_accuracy"] == 100.0 and scores["slot_micro_f1"] == 100.0

[train | gold tags] Intent accuracy: 100.00% | Slot micro-F1: 100.00% (P 100.00%, R 100.00%) | n=11,514
[dev | gold tags] Intent accuracy: 100.00% | Slot micro-F1: 100.00% (P 100.00%, R 100.00%) | n=2,033
[test | gold tags] Intent accuracy: 100.00% | Slot micro-F1: 100.00% (P 100.00%, R 100.00%) | n=2,974


### 3.2 What the later sections use

| object | use |
|---|---|
| `data["train" / "dev" / "test"]` | preprocessed examples (`token_ids`, `tag_ids`, `intent_id`, `gold_slots`, …) |
| `vocab` (`len(vocab)`, `vocab.pad_id`, `vocab.itos`) | word vocabulary from train |
| `intent_itos` / `intent_stoi`, `tag_itos` / `tag_stoi` | label maps (60 intents, 111 bio tags) |
| `PAD_TAG_ID` (-100) | ignore index for the slot loss |
| `make_loader(split, batch_size, shuffle)` | padded batches |
| `make_prediction_from_ids(example, intent_id, tag_ids)` | model ids → `{"intent", "slots"}` |
| `evaluate(examples, predictions)`, `format_scores(scores, split)` | the shared scorer |

Typical evaluation loop for a BIO-tagging model (the model itself is defined in later sections):

```python
predictions = [None] * len(data["dev"])
model.eval()
with torch.no_grad():
    for batch in make_loader("dev", batch_size=64):
        intent_logits, slot_logits = model(batch)          # (B, n_intents), (B, T, n_tags)
        for i, intent_id, tag_ids in zip(batch["indices"].tolist(),
                                         intent_logits.argmax(-1), slot_logits.argmax(-1)):
            predictions[i] = make_prediction_from_ids(data["dev"][i], intent_id, tag_ids)
scores = evaluate(data["dev"], predictions)
print(format_scores(scores, "dev"))
```

## Model
Model Test. WIP

In [18]:
import time
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# seed the weight init and dropout too (SEED already fixes the shuffle order)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# baseline hyperparameters
EMB_DIM    = 300
HIDDEN_DIM = 256    # per direction
NUM_LAYERS = 2
DROPOUT    = 0.3
LR         = 1e-3
BATCH_SIZE = 32
EPOCHS     = 30
PATIENCE   = 5      # stop after this many epochs without a better dev score
CLIP_NORM  = 5.0
print(f"device: {DEVICE}")

device: cuda


In [ ]:
class JointBiLSTM(nn.Module):
    """one shared bilstm encoder with two heads: a bio tag per token, and an intent from the max-pooled states."""

    def __init__(self, vocab_size, n_intents, n_tags, emb_dim, hidden_dim, num_layers, dropout, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.slot_head = nn.Linear(2 * hidden_dim, n_tags)
        self.intent_head = nn.Linear(2 * hidden_dim, n_intents)

    def forward(self, batch):
        x = self.dropout(self.embedding(batch["input_ids"]))
        packed = pack_padded_sequence(x, batch["lengths"].cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))  # back to (B, T, 2H)
        out = self.dropout(out)

        slot_logits = self.slot_head(out)                                           # (B, T, n_tags)
        mask = batch["attention_mask"].unsqueeze(-1)
        pooled = out.masked_fill(~mask, float("-inf")).max(dim=1).values            # padding never wins the max
        intent_logits = self.intent_head(pooled)                                    # (B, n_intents)
        return intent_logits, slot_logits

def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def predict(model, split, batch_size=64):
    """run the model over one split and give back predictions in the shared format, in split order."""
    model.eval()
    predictions = [None] * len(data[split])
    for batch in make_loader(split, batch_size=batch_size):
        intent_logits, slot_logits = model(to_device(batch))
        for i, intent_id, tag_ids in zip(batch["indices"].tolist(),
                                         intent_logits.argmax(-1).tolist(), slot_logits.argmax(-1).tolist()):
            predictions[i] = make_prediction_from_ids(data[split][i], intent_id, tag_ids)
    return predictions


In [20]:
model = JointBiLSTM(len(vocab), len(intent_itos), len(tag_itos), EMB_DIM, HIDDEN_DIM,
                    NUM_LAYERS, DROPOUT, vocab.pad_id).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"total parameters: {n_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
intent_loss_fn = nn.CrossEntropyLoss()
slot_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_TAG_ID)
train_loader = make_loader("train", batch_size=BATCH_SIZE, shuffle=True)

best_score, best_epoch, best_state, history = -1.0, 0, None, []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, start = 0.0, time.time()
    for batch in train_loader:
        batch = to_device(batch)
        intent_logits, slot_logits = model(batch)
        loss = (intent_loss_fn(intent_logits, batch["intent_labels"])
                + slot_loss_fn(slot_logits.reshape(-1, slot_logits.size(-1)), batch["slot_labels"].reshape(-1)))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optimizer.step()
        total_loss += loss.item() * len(batch["indices"])

    dev_scores = evaluate(data["dev"], predict(model, "dev"))
    train_loss = total_loss / len(data["train"])
    history.append({"epoch": epoch, "train_loss": train_loss, **dev_scores})
    print(f"epoch {epoch:>2} | loss {train_loss:.4f} | {format_scores(dev_scores, 'dev')} | {time.time() - start:.1f}s")

    # keep the checkpoint with the best dev combined score
    if dev_scores["combined"] > best_score:
        best_score, best_epoch = dev_scores["combined"], epoch
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    elif epoch - best_epoch >= PATIENCE:
        print(f"no dev improvement for {PATIENCE} epochs, stopping")
        break

model.load_state_dict(best_state)
print(f"\nbest epoch: {best_epoch} (dev combined {best_score:.2f})")


total parameters: 4,398,667
epoch  1 | loss 3.3683 | [dev] Intent accuracy: 71.13% | Slot micro-F1: 41.40% (P 46.55%, R 37.28%) | n=2,033 | 7.7s
epoch  2 | loss 1.4678 | [dev] Intent accuracy: 79.19% | Slot micro-F1: 56.41% (P 58.04%, R 54.87%) | n=2,033 | 7.4s
epoch  3 | loss 0.9723 | [dev] Intent accuracy: 81.80% | Slot micro-F1: 61.56% (P 58.54%, R 64.91%) | n=2,033 | 7.2s
epoch  4 | loss 0.7063 | [dev] Intent accuracy: 83.57% | Slot micro-F1: 66.75% (P 64.82%, R 68.79%) | n=2,033 | 7.2s
epoch  5 | loss 0.5370 | [dev] Intent accuracy: 83.03% | Slot micro-F1: 67.29% (P 63.69%, R 71.32%) | n=2,033 | 7.3s
epoch  6 | loss 0.4296 | [dev] Intent accuracy: 83.52% | Slot micro-F1: 69.86% (P 68.00%, R 71.82%) | n=2,033 | 7.0s
epoch  7 | loss 0.3421 | [dev] Intent accuracy: 83.62% | Slot micro-F1: 69.81% (P 66.47%, R 73.51%) | n=2,033 | 6.7s
epoch  8 | loss 0.2842 | [dev] Intent accuracy: 83.57% | Slot micro-F1: 71.79% (P 69.22%, R 74.55%) | n=2,033 | 7.5s
epoch  9 | loss 0.2505 | [dev] Inten

In [21]:
SAVE_PATH = Path("checkpoints") / "baseline_bilstm_2layers.pt"
SAVE_PATH.parent.mkdir(exist_ok=True)

torch.save({
    "model_state": best_state,
    "config": {"emb_dim": EMB_DIM, "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS, "dropout": DROPOUT},
    "best_epoch": best_epoch,
    "best_dev_combined": best_score,
    "history": history,
    # vocab and label order, so the ids still line up if preprocessing ever changes
    "vocab_itos": vocab.itos,
    "intent_itos": intent_itos,
    "tag_itos": tag_itos,
}, SAVE_PATH)
print(f"saved to {SAVE_PATH.as_posix()}")


saved to checkpoints/baseline_bilstm_2layers.pt


In [ ]:
CKPT_PATH = Path("checkpoints") / "baseline_bilstm_2layers.pt"

def load_checkpoint(path):
    """rebuild the model from a saved checkpoint and load its weights."""
    ckpt = torch.load(path, map_location=DEVICE)
    # make sure the ids mean the same thing as when it was trained
    assert ckpt["vocab_itos"] == vocab.itos and ckpt["intent_itos"] == intent_itos and ckpt["tag_itos"] == tag_itos
    cfg = ckpt["config"]
    m = JointBiLSTM(len(vocab), len(intent_itos), len(tag_itos), cfg["emb_dim"], cfg["hidden_dim"],
                    cfg["num_layers"], cfg["dropout"], vocab.pad_id).to(DEVICE)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    return m, ckpt

m, ckpt = load_checkpoint(CKPT_PATH)
print(f"{CKPT_PATH.as_posix()} | {ckpt['config']['num_layers']} layer(s) | best epoch {ckpt['best_epoch']} | "
      f"{sum(p.numel() for p in m.parameters()):,} params")
for split in ("dev", "test"):
    print(format_scores(evaluate(data[split], predict(m, split)), split))

pd.DataFrame(ckpt["history"]).set_index("epoch")[["train_loss", "intent_accuracy", "slot_micro_f1"]]

checkpoints/baseline_bilstm_2layers.pt | 2 layer(s) | best epoch 26 | 4,398,667 params
[dev] Intent accuracy: 84.55% | Slot micro-F1: 75.47% (P 73.73%, R 77.29%) | n=2,033
[test] Intent accuracy: 83.36% | Slot micro-F1: 73.06% (P 71.17%, R 75.06%) | n=2,974


,train_loss,intent_accuracy,slot_micro_f1
epoch,,,
1,3.368327,71.126414,41.402153
2,1.467770,79.193310,56.412877
3,0.972306,81.800295,61.560217
4,0.706292,83.571077,66.747046
5,0.537028,83.030005,67.291911
6,0.429609,83.521889,69.857385
7,0.342140,83.620266,69.813547
8,0.284244,83.571077,71.787509
9,0.250486,83.521889,73.370219
